# 终结性考核：智慧路口车流数据分析与信号配时决策

> **考核性质**：终结性、闭卷上机
>
> **建议时长**：**90 分钟**（含读题 5 分钟、编码 70 分钟、自检 10 分钟、提交 5 分钟）
>
> **总分**：100 分
>
> **考核范围**：第 1—12 章（变量与数据类型、流程控制、字符串、容器、函数、高级函数、文件操作、异常处理、模块、面向对象）。**不含** turtle 与 pygame，以保证 90 分钟内可完成。

---

## 本卷结构

| 阶段 | 内容 | 章节位置 | 作用 |
|---|---|---|---|
| 第一部分 | 考核说明与知识点对照 | 一、二节 | 数据背景、任务清单、与各章的对应关系 |
| 第二部分 | 考核任务 | 任务一 — 任务五 | 现场按序编码，逐项自检 |
| 第三部分 | 提交要求 | 三节 | 交什么、怎么交、考场纪律 |
| 第四部分 | 评分要点与预期结果 | 四、五节 | 教师阅卷口径、学生自检对照 |
| 第五部分 | 参考答案与解析 | 六、七节 | **交卷后**再对照，按任务看思路与易错点 |

## 考核知识点对照表

| 任务 | 分值 | 主要知识点 | 对应章节 |
|---|---|---|---|
| 任务一　数据读入与清洗 | 20 | `with open` + `encoding`、`split()`、`try-except`、`continue` 跳过 | 第 8、9 章 |
| 任务二　分时段统计与判定 | 30 | 字典累加 `get(k, 0)`、`max()` / `min()` + `key`、函数与多分支、`f-string` 对齐 | 第 3、4、5 章 |
| 任务三　信号配时决策 | 30 | 类与 `__init__`、字典映射、`__str__`、继承与 `super()`、方法重写 | 第 11、12 章 |
| 任务四　决策报表输出 | 10 | `w` 模式写入、`",".join()`、读回验证行数 | 第 8 章 |
| 任务五　代码规范与文档 | 10 | 文档字符串、命名规范、`if __name__ == "__main__"` | 第 6、10 章 |

> **命题原则**：本卷**不引入任何新语法**，所有考点均在前 12 章的代码示范或练习中出现过。若某一步不会写，应回到对应章节的案例重新读一遍，而不是临时上网找代码。

---

# 第一部分　考核说明

## 一、考核背景与给定数据

某市交通运行监测中心采集了早高峰 6 个 2 小时时段、2 条车道共 14 条记录（其中 4 条为脏数据），字段为：`时段,车道,车流量,平均车速`。数据由不同设备上报，**存在三类脏数据**：空值、非数字、数值越界。

现要求你编写一个完整的 Python 程序，完成"读入 → 清洗 → 统计 → 决策 → 出报表"的闭环，为信号配时提供依据。

**给定数据文件 `traffic_data.csv`（随考卷下发，含表头共 15 行）：**

```csv
时段,车道,车流量,平均车速
06:00-08:00,1,820,68
06:00-08:00,2,760,72
08:00-10:00,1,1100,42
08:00-10:00,2,1050,45
10:00-12:00,1,760,58
10:00-12:00,2,,60
12:00-14:00,1,690,62
12:00-14:00,2,720,64
14:00-16:00,1,980,48
14:00-16:00,2,abc,50
16:00-18:00,1,1650,32
16:00-18:00,2,1580,30
10:00-12:00,3,760
12:00-14:00,3,1500,320
```

> 最后两行是"设备补报"的异常行：倒数第 2 行缺 1 个字段；最后一行车速 320 km/h 超出物理范围。

**三类脏数据的判别口径（务必先读懂再动手）：**

| 类型 | 判别依据 | 本卷出现的行 |
|---|---|---|
| 格式异常 | 按 `,` 切分后字段数 ≠ 4 | 倒数第 2 行（仅 3 个字段） |
| 数值异常 | 字段数正确，但 `int()` 抛 `ValueError`（含空串） | `10:00-12:00,2,,60`、`14:00-16:00,2,abc,50` |
| 越界异常 | 转换成功，但数值超出合理范围 | `12:00-14:00,3,1500,320`（车速 320） |

---

# 第二部分　考核任务

## 二、考核任务（共 5 项，含规范分）

### 任务一　数据读入与清洗（20 分）

**要求：**

1. 用 `with open(..., encoding="utf-8")` 读取 `traffic_data.csv`，跳过表头；
2. 逐行 `strip()` 后按 `,` 切分，字段数为 4；**字段数不对的记录计入"格式异常"**；
3. 对车流量与车速做 `int()` 转换，捕获 `ValueError`，失败计入"数值异常"（**空字符串 `int("")` 同样抛出 `ValueError`，一并计入数值异常**）；
4. 车速合法范围 0~300、车流量合法范围 0~3000，**转换成功但越界**的计入"越界异常"；
5. 每条异常记录**必须打印原因**（含行号），且**不得中断程序**；
6. 全部处理结束后输出：有效记录数、格式异常数、数值异常数、越界异常数。

**考核点：** `with open` + `encoding`、`split`、`try-except`、`continue` 跳过、计数器累加。

---

### 任务二　分时段统计与拥堵判定（30 分）

**要求：**

1. 按"时段"汇总有效记录的车流量（同一时段两条车道要**相加**），存入字典；
2. 计算全天总流量、有效记录的平均车速（保留 1 位小数）；
3. 找出**流量最大的时段**与**车速最低的时段**；
4. 编写函数 `judge(flow, speed)`，按下列规则返回拥堵等级：
   - `speed >= 55` 且 `flow < 1500` → `"畅通"`
   - `speed >= 40` → `"缓行"`
   - `speed >= 25` → `"拥堵"`
   - 其余 → `"严重拥堵"`
5. 输出每个时段的"流量 / 平均车速 / 等级"三列表格（用 `f-string` 对齐）。

**考核点：** 字典累加 `get(k, 0)`、`max()` 配合 `key`、函数定义与多分支判定、字符串格式化对齐。

---

### 任务三　信号配时决策（30 分）

**要求：**

1. 定义类 `SignalPlan`，构造时接收 `period`（时段）、`flow`（流量）、`level`（等级）；
2. 类中实现 `green_time()`：按下表返回绿灯时长（秒），要求用**字典映射**而非 `if` 链；

   | 等级 | 绿灯时长 |
   |---|---|
   | 畅通 | 30 |
   | 缓行 | 50 |
   | 拥堵 | 70 |
   | 严重拥堵 | 90 |

3. 类中实现 `cycle()`：周期 = 绿灯 + 黄灯 3 + 红灯 40；**若周期超过 120 秒，则红灯被压缩、周期取 120 秒**（绿灯与黄灯不变，即 `min(绿灯+3+40, 120)`）；
4. 类中实现 `__str__`，返回形如 `16:00-18:00 流量 3230 拥堵 绿灯 70s 周期 113s`；
5. 为每个时段创建一个 `SignalPlan` 对象，存入列表并逐个打印；
6. 用**继承**扩展一个 `EmergencyPlan` 子类：当等级为"严重拥堵"时，绿灯在父类基础上再增加 5 秒（重写 `green_time()`，用 `super()` 取父类值）。

**考核点：** 类定义、`__init__`、字典映射、方法计算、`__str__`、继承与 `super()`、方法重写。

---

### 任务四　决策报表输出（10 分）

**要求：**

1. 将每个时段的决策结果写入 `signal_plan.csv`，表头为 `时段,流量,等级,绿灯,周期`；
2. 文件末尾追加一行汇总：`合计,10110,,,`（第 2 列写全天总流量，后 3 列留空）；
3. 写入后再读回，打印文件总行数，验证写入成功；
4. 全程使用 `encoding="utf-8"`。

**考核点：** `w` 模式写入、`",".join()` 拼接行、汇总行追加到文件尾、读回验证行数。

---

### 任务五　代码规范与文档（10 分）

1. 每个函数/类必须有文档字符串，说明输入、输出与职责；
2. 变量命名见名知义（禁止 `a`、`b`、`tmp1`）；
3. 关键逻辑有中文注释；
4. 程序可直接运行、无语法错误、无未捕获异常；
5. 顶层代码用 `if __name__ == "__main__":` 组织。

---

# 第三部分　提交要求

## 三、提交要求与考场纪律

1. 提交 `answer.py` 与运行生成的 `signal_plan.csv`；
2. 文件名格式：`学号_姓名_终结性考核.py`；
3. 允许查阅教材与官方文档，**不得联网搜索现成代码**，不得互相讨论；
4. 程序必须能在考场机器的 Python 3 环境下直接运行（数据文件与该程序放在同一目录）。

**交卷前必须完成的三件事：**

- [ ] 把 `traffic_data.csv` 与 `answer.py` 放在同一目录，删除旧 `signal_plan.csv` 后**重跑一次**；
- [ ] 确认四个计数与预期值一致（有效 10、格式 1、数值 2、越界 1），若不则回头查清洗逻辑；
- [ ] 确认终端无红色 `Traceback`，`signal_plan.csv` 共 8 行。

---

# 第四部分　评分要点与预期结果

## 四、评分要点与时间分配

### 4.1 评分细则

| 项目 | 分值 | 评分细则 | 扣分点 |
|---|---|---|---|
| 任务一 数据读入与清洗 | 20 | 正确读文件 5；字段切分与表头跳过 4；三类异常分别捕获并计数 6；异常不中断程序 3；汇总输出 2 | 未写 `encoding` 扣 2；异常未捕获导致程序崩溃，本项不超过 8 分 |
| 任务二 统计与判定 | 30 | 字典按时段累加 8；总量/均值/峰值时段 8；`judge()` 四分支正确 8；三列对齐输出 6 | 同车道未相加（直接覆盖）扣 6；`judge()` 分支顺序错误扣 4 |
| 任务三 配时决策 | 30 | 类结构与 `__init__` 6；`green_time()` 用字典映射 6；`cycle()` 含压缩逻辑 6；`__str__` 格式正确 4；`EmergencyPlan` 继承与 `super()` 6；对象列表输出 2 | 用 `if` 链实现映射扣 3；未用 `super()` 扣 3；周期压缩公式错误扣 4 |
| 任务四 报表输出 | 10 | 表头与数据行 4；汇总行 2；读回验证 2；编码正确 2 | 用 `w` 覆盖导致数据丢失扣 3 |
| 任务五 规范与文档 | 10 | 文档字符串 3；命名规范 3；注释 2；可运行 2 | 程序无法运行，本项计 0 |
| **合计** | **100** | | |

### 4.2 时间分配建议

| 阶段 | 时间 | 检查点 |
|---|---|---|
| 读题与数据核对 | 5 min | 确认 14 行数据、明确 3 类异常 |
| 任务一 | 20 min | 打印 4 个计数，与预期值核对 |
| 任务二 | 20 min | 打印三列表格 |
| 任务三 | 25 min | 打印 6 行 `SignalPlan` + 1 行 `EmergencyPlan` |
| 任务四 | 10 min | 生成 `signal_plan.csv` |
| 自检与提交 | 10 min | 重跑一次，确认无异常 |

## 五、预期正确结果（阅卷对照）

| 项目 | 预期值 |
|---|---|
| 有效记录 | **10** 条 |
| 格式异常 | **1** 条（缺字段） |
| 数值异常 | **2** 条（空值与 `abc`） |
| 越界异常 | **1** 条（车速 320） |
| 全天总流量 | **10110** 辆（820+760+1100+1050+760+690+720+980+1650+1580） |
| 时段汇总 | 06-08 = 1580，08-10 = 2150，10-12 = 760，12-14 = 1410，14-16 = 980，16-18 = **3230**（峰值） |
| 平均车速 | 68,72,42,45,58,62,64,48,32,30 → 均值 **52.1** |
| 车速最低时段 | **16:00-18:00**（32、30，平均 31.0） |
| 等级判定 | 06-08 **缓行**、08-10 **缓行**、10-12 **畅通**、12-14 **畅通**、14-16 **缓行**、16-18 **拥堵** |
| 配时（绿灯 / 周期） | 缓行 50/93、畅通 30/73、拥堵 70/113；16-18 为 70/113 |
| 应急方案 | `16:00-18:00 流量 3230 严重拥堵 绿灯 95s 周期 120s`（95+3+40=138 超过 120，周期触顶） |
| `signal_plan.csv` | 共 **8** 行（表头 1 + 数据 6 + 合计 1） |

> **最易错处**：06-08 时段。车速 70 km/h 看似"畅通"，但流量 1580 已超过 1500，按 `speed >= 55 and flow < 1500` 必须落到"缓行"。学生若把 `judge()` 写成只看车速，此处即会出错，并连带影响配时（绿灯由 50 变 30）。

---

# 第五部分　参考答案与解析

> **使用建议**：先完成第三部分的自检清单，再对照本部分。重点看**每个任务"为什么这样写"**与「常见错误」，不必逐字比对代码。

## 六、参考答案（完整代码）

In [ ]:
"""
终结性考核参考答案
课程：Python 编程基础（智慧交通）
功能：读入路口车流数据 -> 清洗 -> 统计 -> 配时决策 -> 输出报表
"""


# ============ 任务一：读入与清洗 ============
def load_data(filename):
    """读取 CSV 并清洗，返回 (有效记录列表, 异常计数字典)"""
    valid = []
    errors = {"格式异常": 0, "数值异常": 0, "越界异常": 0}

    with open(filename, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for i in range(1, len(lines)):          # 跳过表头
        line = lines[i].strip()
        if line == "":
            continue
        parts = line.split(",")

        if len(parts) != 4:
            errors["格式异常"] += 1
            print("  第" + str(i + 1) + "行 格式异常：字段数为 " + str(len(parts)))
            continue

        try:
            flow = int(parts[2])
            speed = int(parts[3])
        except ValueError:
            errors["数值异常"] += 1
            print("  第" + str(i + 1) + "行 数值异常：" + parts[2] + " / " + parts[3])
            continue

        if flow < 0 or flow > 3000 or speed < 0 or speed > 300:
            errors["越界异常"] += 1
            print("  第" + str(i + 1) + "行 越界异常：流量 " + str(flow) + "，车速 " + str(speed))
            continue

        valid.append((parts[0], parts[1], flow, speed))

    print("有效记录：", len(valid), "条")
    for k in errors:
        print("  " + k + "：", errors[k], "条")
    return valid, errors


# ============ 任务二：统计与判定 ============
def judge(flow, speed):
    """拥堵等级判定"""
    if speed >= 55 and flow < 1500:
        return "畅通"
    elif speed >= 40:
        return "缓行"
    elif speed >= 25:
        return "拥堵"
    return "严重拥堵"


def summarize(records):
    """按时间汇总流量、计算平均车速，返回 (时段流量字典, 时段车速字典, 总量, 均速)"""
    flow_by_period = {}
    speed_list = []
    speed_by_period = {}

    for period, lane, flow, speed in records:
        flow_by_period[period] = flow_by_period.get(period, 0) + flow
        speed_list.append(speed)
        speed_by_period.setdefault(period, []).append(speed)

    total_flow = sum(flow_by_period.values())
    avg_speed = round(sum(speed_list) / len(speed_list), 1)

    for p in speed_by_period:
        vals = speed_by_period[p]
        speed_by_period[p] = round(sum(vals) / len(vals), 1)

    return flow_by_period, speed_by_period, total_flow, avg_speed


# ============ 任务三：配时决策 ============
class SignalPlan:
    """单个时段的信号配时方案"""

    GREEN_MAP = {"畅通": 30, "缓行": 50, "拥堵": 70, "严重拥堵": 90}
    YELLOW = 3
    RED = 40
    MAX_CYCLE = 120

    def __init__(self, period, flow, level):
        self.period = period
        self.flow = flow
        self.level = level

    def green_time(self):
        """按等级查表得到绿灯时长"""
        return SignalPlan.GREEN_MAP[self.level]

    def cycle(self):
        """周期 = 绿 + 黄 + 红；超过 120 秒则红灯被压缩，周期取 120"""
        raw = self.green_time() + SignalPlan.YELLOW + SignalPlan.RED
        return min(raw, SignalPlan.MAX_CYCLE)

    def __str__(self):
        return (self.period + " 流量 " + str(self.flow) + " " + self.level
                + " 绿灯 " + str(self.green_time()) + "s 周期 " + str(self.cycle()) + "s")


class EmergencyPlan(SignalPlan):
    """应急方案：严重拥堵时绿灯再加 5 秒"""

    def green_time(self):
        base = super().green_time()
        if self.level == "严重拥堵":
            return base + 5
        return base


# ============ 任务四：报表输出 ============
def write_report(filename, plans, total_flow):
    """写出配时报表"""
    with open(filename, "w", encoding="utf-8") as f:
        f.write("时段,流量,等级,绿灯,周期\n")
        for p in plans:
            row = [p.period, str(p.flow), p.level, str(p.green_time()), str(p.cycle())]
            f.write(",".join(row) + "\n")
        f.write("合计," + str(total_flow) + ",,,\n")

    with open(filename, "r", encoding="utf-8") as f:
        count = len(f.readlines())
    print("报表已写入 " + filename + "，共 " + str(count) + " 行")


# ============ 主程序 ============
if __name__ == "__main__":
    print("=== 任务一：数据读入与清洗 ===")
    records, errors = load_data("traffic_data.csv")

    print()
    print("=== 任务二：分时段统计与等级判定 ===")
    flow_by_period, speed_by_period, total_flow, avg_speed = summarize(records)
    print("全天总流量：", total_flow, "辆 | 平均车速：", avg_speed, "km/h")

    peak_period = max(flow_by_period, key=lambda p: flow_by_period[p])
    slow_period = min(speed_by_period, key=lambda p: speed_by_period[p])
    print("流量最大时段：", peak_period, flow_by_period[peak_period], "辆")
    print("车速最低时段：", slow_period, speed_by_period[slow_period], "km/h")
    print()
    print(f"{'时段':<14}{'流量':>8}{'车速':>8}{'等级':>10}")
    for p in flow_by_period:
        lv = judge(flow_by_period[p], speed_by_period[p])
        print(f"{p:<14}{flow_by_period[p]:>8}{speed_by_period[p]:>8}{lv:>10}")

    print()
    print("=== 任务三：信号配时决策 ===")
    plans = []
    for p in flow_by_period:
        lv = judge(flow_by_period[p], speed_by_period[p])
        plans.append(SignalPlan(p, flow_by_period[p], lv))
    for plan in plans:
        print(plan)

    print()
    print("--- 应急方案（继承演示）---")
    emergency = EmergencyPlan("16:00-18:00", 3230, "严重拥堵")
    print(emergency)

    print()
    print("=== 任务四：报表输出 ===")
    write_report("signal_plan.csv", plans, total_flow)

### 6.1 任务一解析：读入与清洗

| 代码段 | 作用 | 对应知识点 |
|---|---|---|
| `with open(filename, "r", encoding="utf-8") as f` | 打开文件并保证自动关闭；显式 UTF-8 避免中文乱码 | 8.4 `with` 与编码 |
| `lines = f.readlines()` | 一次读入全部行，便于按行号报错 | 8.2 三种读取方式 |
| `for i in range(1, len(lines))` | 从下标 1 开始，跳过第 0 行表头 | 8.2 逐行处理 |
| `if line == "": continue` | 跳过空行，避免无意义的"格式异常" | 9.5 容错跳过 |
| `if len(parts) != 4:` | 先判字段数，再转换数字——顺序不能颠倒 | 4.4 `split()` |
| `try: flow = int(parts[2]) ... except ValueError` | 空串与 `abc` 都会抛 `ValueError`，一并捕获 | 9.2 `try-except` |
| `if flow < 0 or flow > 3000 or speed < 0 or speed > 300` | 转换成功后再判越界，与数值异常分开计数 | 2.5 逻辑运算 |
| `errors = {"格式异常": 0, ...}` | 用字典统一计数，避免 4 个散落的变量 | 5.5 字典 |
| 三处 `continue` | 坏数据**跳过但不中断**整批处理，这是本题的容错骨架 | 9.5 容错模式 |

### 6.2 任务二解析：统计与判定

| 代码段 | 作用 | 对应知识点 |
|---|---|---|
| `flow_by_period[period] = flow_by_period.get(period, 0) + flow` | 同一时段两条车道**相加**，`get(p, 0)` 解决键不存在的问题 | 5.5 字典累加 |
| `speed_by_period.setdefault(period, []).append(speed)` | 先按时间段分组收集车速，再求均值（不能直接平均平均） | 5.5 `setdefault` |
| `round(sum(speed_list) / len(speed_list), 1)` | 有效记录的总体均速，保留 1 位小数 | 2.6 `round()` |
| `max(flow_by_period, key=lambda p: flow_by_period[p])` | 取"值最大的键"，直接 `max(字典)` 只会比较键名 | 7.3 `sorted/max` 的 `key` |
| `judge()` 四分支顺序 | 先判"畅通"（含流量条件），再由高到低判车速，顺序错则结果错 | 3.1 `if-elif-else` |
| `f"{p:<14}{flow:>8}{speed:>8}{lv:>10}"` | 四列定宽对齐，中英文混排时留足宽度 | 4.5 `f-string` 对齐 |

> **易错点**：把 `judge()` 写成先判 `speed >= 40` 再判"畅通"，则 06-08（车速 70、流量 1580）会被判成缓行…**看似对，但 10-12（车速 58、流量 760）也会被判成缓行**，正确结果应为畅通。分支顺序必须"条件最严的放最前"。

### 6.3 任务三解析：配时决策

| 代码段 | 作用 | 对应知识点 |
|---|---|---|
| `class SignalPlan:` + `__init__(self, period, flow, level)` | 把"一个时段的方案"封装为对象，属性即业务字段 | 11.1、11.2 |
| `GREEN_MAP = {"畅通": 30, ...}` | 类属性字典映射，替代 4 段 `if-elif`，题目明确要求 | 11.3、5.5 |
| `return SignalPlan.GREEN_MAP[self.level]` | 通过类名访问类属性，语义清晰 | 11.3 |
| `min(raw, SignalPlan.MAX_CYCLE)` | 周期封顶 120 秒，等价于"红灯被压缩" | 2.4 `min()` |
| `def __str__(self):` | 定义 `print(对象)` 时的显示格式，无需手动拼 `info()` | 11.4 `__str__` |
| `class EmergencyPlan(SignalPlan):` | 继承父类全部能力，只改需要变的部分 | 12.2 继承 |
| `base = super().green_time()` | 先取父类结果再 +5，避免把映射表复制一遍 | 12.2 `super()` |
| `if self.level == "严重拥堵": return base + 5` | 条件性重写：只对严重拥堵生效 | 12.3 方法重写 |
| 统一 `print(plan)` | 父类与子类对象走同一接口，输出各自结果——这就是多态 | 12.4 多态 |

### 6.4 任务四解析：报表输出

| 代码段 | 作用 | 对应知识点 |
|---|---|---|
| `with open(filename, "w", encoding="utf-8")` | `w` 模式重建文件，保证每次运行不叠加旧内容 | 8.1 写入模式 |
| `f.write("时段,流量,等级,绿灯,周期\n")` | 表头单独写，注意行尾 `\n` | 8.3 写入换行 |
| `",".join(row)` | 把 5 个字段拼成一行 CSV，比 `+` 拼接更稳 | 4.4 `join()` |
| `f.write("合计," + str(total_flow) + ",,,\n")` | 汇总行后 3 列留空，逗号数量必须与表头一致 | 8.5 CSV 手工写入 |
| 读回 `len(f.readlines())` | 用"读回"验证写入结果，而不是凭感觉 | 8.2 读回校验 |

## 七、常见错误与复核清单

### 7.1 按现象查原因

| 现象 | 原因 | 修正 |
|---|---|---|
| 程序在读数据时崩溃 | 只写了 `try` 包数字转换，`parts[3]` 越界（字段不足）未被处理 | 先判 `len(parts)`，再转换 |
| 异常计数为 0 | 把 `continue` 写在计数之前 | 先计数、再 `continue` |
| 格式异常被算成数值异常 | 未先判字段数，直接 `int(parts[2])` 触发 `IndexError` | 顺序：字段数 → 数值 → 越界 |
| 总流量偏小 | 同一时段两条车道被覆盖（用了 `=` 而非 `+=`） | 用 `get(p, 0) + flow` |
| 峰值时段取成字符串 | `max(flow_by_period)` 比较的是键名 | 加 `key=lambda p: flow_by_period[p]` |
| 等级判定与预期不符 | `judge()` 分支顺序错误，或漏掉 `flow < 1500` | 最严条件放最前 |
| 周期都是 120 | 写成 `max(...)` 或先压缩绿灯 | 用 `min(绿灯+3+40, 120)` |
| `print(plan)` 显示内存地址 | 未定义 `__str__` | 补 `__str__` |
| 子类绿灯没加 5 | 重写时没调 `super()`，或条件写成 `!=` | 先 `super().green_time()` 再判断 |
| 报表行数不是 8 | 汇总行逗号数量不对，或 `w` 与 `a` 用反 | 核对 `,,,` 三个逗号 |
| 中文乱码 | 读写未写 `encoding="utf-8"` | 每处 `open()` 都补上 |

### 7.2 交卷前复核清单

- [ ] 四个计数与预期一致：有效 10 / 格式 1 / 数值 2 / 越界 1；
- [ ] 总流量 10110，均速 52.1，峰值时段 16:00-18:00；
- [ ] 六行等级依次为：缓行、缓行、畅通、畅通、缓行、拥堵；
- [ ] 配时输出中 16-18 为"绿灯 70s 周期 113s"；
- [ ] 应急方案为"绿灯 95s 周期 120s"（体现继承 + 周期封顶）；
- [ ] `signal_plan.csv` 8 行，末行为 `合计,10110,,,`；
- [ ] 每个函数/类都有文档字符串，无 `a`、`b`、`tmp1` 之类命名；
- [ ] 顶层代码在 `if __name__ == "__main__":` 之下。

---

## 课程小结：本卷覆盖的能力

| 能力 | 本卷体现 | 对应章节 |
|---|---|---|
| 读写文本文件、处理编码 | 任务一、任务四 | 第 8 章 |
| 坏数据不崩溃、分类计数 | 任务一 | 第 9 章 |
| 字典累加与分组统计 | 任务二 | 第 5 章 |
| 函数封装与多分支判定 | 任务二 | 第 3、6 章 |
| 格式化对齐输出 | 任务二 | 第 4 章 |
| 类、属性、方法与 `__str__` | 任务三 | 第 11 章 |
| 继承、`super()` 与方法重写 | 任务三 | 第 12 章 |
| 代码规范与文档 | 任务五 | 第 6、10 章 |

> 完成本卷即意味着具备"把一批原始数据变成可交付报表"的完整能力——这也是智慧交通岗位最基础的编程要求。若某任务失分，建议回到对应章节重做该章的 2 个练习，再做本卷的同一任务。

---

[🏠 课程首页](../00_课程索引.ipynb)&nbsp;&nbsp;|&nbsp;&nbsp;[◀ 上一章：本地AI综合实战](../15本地AI综合实战/15_本地AI综合实战.ipynb)